# 1. read dde from file


In [118]:
import pandas as pd
# Try reading the file with space as a delimiter
df = pd.read_csv('data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])
# Split the 'Mutation' column into 'First_mutation' and 'Second_mutation'
df[['First_mutation', 'Second_mutation']] = df['Mutation'].str.split('-', expand=True)
# Display the DataFrame
print(df)


/var/folders/17/rj19bvws2qscyfjmb7m44zmm0000gn/T/ipykernel_62481/111636545.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv('data/kn.her2.all', delim_whitespace=True, header=None, names=['Mutation', 'DDE','DE_double', 'First_DE', 'Second_DE'])


           Mutation       DDE  DE_double  First_DE  Second_DE First_mutation  \
0         D39A-D40A  0.233301 -12.552485 -4.253487  -8.739039           D39A   
1         D39A-D40B  0.777221  -9.559998 -4.253487  -5.508154           D39A   
2         D39A-D40C  0.925157  -9.106178 -4.253487  -5.010899           D39A   
3         D39A-D41A  0.133860 -11.640151 -4.253487  -7.604613           D39A   
4         D39A-D41B  0.386045 -11.713393 -4.253487  -7.495431           D39A   
...             ...       ...        ...       ...        ...            ...   
158197  D225B-C226B -0.016651 -15.816255 -7.860188  -8.096559          D225B   
158198  D225B-C226D  0.079257 -15.383505 -7.860188  -7.669111          D225B   
158199  D225C-C226A  0.228374 -15.461581 -7.357006  -8.181498          D225C   
158200  D225C-C226B  0.080146 -15.296965 -7.357006  -8.096559          D225C   
158201  D225C-C226D  0.259319 -14.947479 -7.357006  -7.669111          D225C   

       Second_mutation  
0             

read from data/top_syn_list.tsv , and traslate the amino acids with data/pr.reduce4.redux, where the first column is postion, second column corrosponding aas translate to A, third to B, and fourth to C, fifth to D and save it locally

In [119]:
import pandas as pd
aa_offset = 0 #redux start with 0, but position start with 1

# Load the TOP_SYN_LIST.TSV file
top_syn_list = pd.read_csv('data/top_syn_list_NNRTI.tsv', sep='\t')

# Load the PR.REDUCE4.REDUX file
with open('data/rt.reduce4.redux', 'r') as f:
    rt_reduce4_redux = f.readlines()

# Parse PR.REDUCE4.REDUX into a dictionary
translation_dict = {}
for line in rt_reduce4_redux:
    parts = line.strip().split(' ')
    position = int(parts[0]) +aa_offset# Convert to 1-based index
    translation_dict[position] = {
        'A': set(parts[1]),
        'B': set(parts[2]),
        'C': set(parts[3]),
        'D': set(parts[4])
    }
untranslated_wildtype_aa1 = top_syn_list['wildtype_aa1']
untranslated_mutate_aa1 = top_syn_list['mutate_aa1']
untranslated_wildtype_aa2 = top_syn_list['wildtype_aa2']
untranslated_mutate_aa2 = top_syn_list['mutate_aa2']
# Function to translate amino acids
def translate_aa(position, aa):
    if position in translation_dict:
        for key, aa_set in translation_dict[position].items():
            if aa in aa_set:
                return key
    return None

# Apply translation to the dataframe
for col in ['wildtype_aa1', 'mutate_aa1', 'wildtype_aa2', 'mutate_aa2']:
    top_syn_list[col] = top_syn_list.apply(lambda row: translate_aa(row['pos1'] if '1' in col else row['pos2'], row[col]), axis=1)

# Look up the DDE value and add it as a new column
def lookup_dde(row):
    mutation1 = f"{row['wildtype_aa1']}{row['pos1']}{row['mutate_aa1']}"
    mutation2 = f"{row['wildtype_aa2']}{row['pos2']}{row['mutate_aa2']}"
    dde_row = df[(df['First_mutation'] == mutation1) & (df['Second_mutation'] == mutation2)]
    return dde_row['DDE'].values[0] if not dde_row.empty else None

top_syn_list['DDE'] = top_syn_list.apply(lookup_dde, axis=1)
# Keep the untranslated amino acids in the dataframe
# Save the translated dataframe

top_syn_list['untranslated_wildtype_aa1'] = untranslated_wildtype_aa1
top_syn_list['untranslated_mutate_aa1'] = untranslated_mutate_aa1
top_syn_list['untranslated_wildtype_aa2'] = untranslated_wildtype_aa2
top_syn_list['untranslated_mutate_aa2'] = untranslated_mutate_aa2

top_syn_list.to_csv('data/translated_top_syn_list.tsv', sep='\t', index=False)

# 2. index each sequence by its mutations compared to in.consensus.reduce4.seq

read in from ../data/in.reduce4.seq, there are 1220 sequences, create a df, with last column as mutations: with a list of mutaiton that occored compared to ../data/in.consensus.reduce4.seq the mutations are in format for example: [D148B, C140D, ...] where D and C notes the wildtime from consensus at pos 148 and 140

In [120]:
# Read the consensus sequence
with open('data/rt.consensus.reduce4.seq', 'r') as f:
    consensus_sequence = f.read().strip()

# Read the 1220 sequences
sequences = []
with open('data/RT.reduce4.seq', 'r') as f:
    for line in f:
        sequences.append(line.strip())

# Create a DataFrame to store sequences and their mutations
sequence_df = pd.DataFrame({'Sequence': sequences})

# Function to identify mutations compared to the consensus sequence
def find_mutations(sequence, consensus):
    mutations = []
    for i, (seq_residue, cons_residue) in enumerate(zip(sequence, consensus), start=1):
        if seq_residue != cons_residue:
            mutations.append(f"{cons_residue}{i}{seq_residue}")
    return mutations

# Add a column for mutations
sequence_df['Mutations'] = sequence_df['Sequence'].apply(lambda seq: find_mutations(seq, consensus_sequence))
sequence_df['Mutations_count'] = sequence_df['Mutations'].apply(len)
# Display the DataFrame
print(sequence_df)

                                                Sequence  \
0      DDDCCCBACCDCADCDBDCCBDABCABAADBCCDCDAACDCBDABD...   
1      ADDCCCBACDDCADCDBDCCBDABCABAADBCCDCDAACDCBDADD...   
2      BDDCCCBACCDCADCDBDCCBDABCABAADBCCDCBAACDCBDABD...   
3      DDDCCCBACDACADCDBDCCBBABCABADDBDCDCDAACDCBDADD...   
4      DDDCCCBACDDCADCDBDCCBDABCABAADBCCDCDAACDCBDABD...   
...                                                  ...   
19189  DDCCCCBACDDCADCDBDCCBBABCABAADBCCDCDAACDCBDADD...   
19190  BDCCDBBACDDCADCDBDCCBBABCABADDBCCDCDDACDCBDADD...   
19191  DDDCBCBACDDCADCDBDCCBBABCABADDBCCDCCAACDCBDADD...   
19192  DCDCCCBACDACADCDBDCCBBABCABAADACCDCDAACDCBDADD...   
19193  BDDCACBACDDCADCDBDCCBDABCABAABBCCDCDAACDCBDADD...   

                                               Mutations  Mutations_count  
0             [D10C, D45B, C65B, C66A, A84C, B85C, A97C]                7  
1                [D1A, C65B, C135B, B136D, D169B, D173C]                6  
2            [D1B, D10C, D36B, D45B, B85C, B139A, D

# 3. J matrix and delta e definition

## 3.1 J matrix

In [121]:
import numpy as np

#dictionary of J matrix
J_dict = {}
# Load the J matrix from the downloaded file
J = np.load('data/J_RT.npy')

row = 0
# Determine the largest position in the 'Mutation' column of the dataframe
min_position = min(
    int(mutation[1:-1]) for mutation in df['First_mutation'].tolist() + df['Second_mutation'].tolist()
)
max_position = max(
    int(mutation[1:-1]) for mutation in df['First_mutation'].tolist() + df['Second_mutation'].tolist()
)
print(min_position, max_position)
# print(max_position+1)
# Update the range to use the largest position
for pos1 in range(min_position, max_position + 1):
    for pos2 in range(pos1 + 1, max_position + 1):
        for i, aa1 in enumerate(['A', 'B', 'C', 'D']):
            for j, aa2 in enumerate(['A', 'B', 'C', 'D']):
                col = i * 4 + j
                J_dict[(pos1, pos2, aa1, aa2)] = J[row, col]
                J_dict[(pos2, pos1, aa2, aa1)] = J[row, col]
        row += 1

print(f"Dictionary created with {len(J_dict)} entries")

39 226
Dictionary created with 562496 entries


## 3.2 define delta e

In [122]:
# Define delta E calculation
def calculate_delta_e(position, old_amino_acid, new_amino_acid, seq, J_dict):
    # E(old_amino_acid)
    energy_old = 0
    for other_pos in range(min_position, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - min_position]  # Access the first sequence in sequence_list
        energy_old += J_dict.get((position, other_pos, old_amino_acid, other_aa), 0)

    # E(new_amino_acid)
    energy_new = 0
    for other_pos in range(min_position, max_position+1):
        if other_pos == position:
            continue
        other_aa = seq[other_pos - min_position]  # Access the first sequence in sequence_list
        energy_new += J_dict.get((position, other_pos, new_amino_acid, other_aa), 0)

    delta_e = energy_old - energy_new
    # print(f"E({old_amino_acid}) at {position}: {energy_old}")
    # print(f"E({new_amino_acid}) at {position}: {energy_new}")
    # print(f"Delta E for {old_amino_acid}{position}{new_amino_acid}: {delta_e}")
    return delta_e

# # Example usage
# position = 140
# old_amino_acid = 'C'
# new_amino_acid = 'D'
# calculate_delta_e(position, old_amino_acid, new_amino_acid, sequence_list, J_dict)

In [123]:
# # Compute D148B on all sequences
# mutation = 'D148B'
# pos, old_aa, new_aa = int(mutation[1:-1]), mutation[0], mutation[-1]

# # List to store results
# results = []

# # Iterate through all sequences
# for index, row in sequence_df.iterrows():
#     sequence = row['Sequence']
#     mutations = row['Mutations']
    
#     # Calculate delta E for D148B
#     de = calculate_delta_e(pos, old_aa, new_aa, sequence, J_dict)
    
#     results.append({
#         'Sequence': sequence,
#         'Mutations': mutations,
#         'DE_D148B': de
#     })

# # Create DataFrame
# results_df = pd.DataFrame(results)

# # Save to TSV
# output_file = 'data/D148B_all_sequences.tsv'
# results_df.to_csv(output_file, sep='\t', index=False)

# print(f"Results saved to {output_file}")
# print(results_df.head())

In [124]:
# define dm12
def calculate_dm12(pos1, old_amino_acid1, new_amino_acid1, pos2, old_amino_acid2, new_amino_acid2, seq, J_dict):
    # Check if positions are already mutated
    current_aa1 = seq[pos1 - min_position]
    current_aa2 = seq[pos2 - min_position]
    
    if current_aa1 != old_amino_acid1 or current_aa2 != old_amino_acid2:
        return None
    
    energy_old = 0
    energy_new = 0

    # old energy
    for other_pos in range(min_position, max_position+1):
        other_aa = seq[other_pos - min_position] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_old += J_dict.get((pos1, pos2, current_aa1, other_aa), 0)
        else:
            energy_old += J_dict.get((pos1, other_pos, old_amino_acid1, other_aa), 0)

    for other_pos in range(min_position, max_position+1):
        other_aa = seq[other_pos - min_position]
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            continue
        else:
            energy_old += J_dict.get((pos2, other_pos, current_aa2, other_aa), 0)

    # new energy
    for other_pos in range(min_position, max_position+1):
        other_aa = seq[other_pos - min_position] 
        if other_pos == pos1:
            continue
        if other_pos == pos2:
            energy_new += J_dict.get((pos1, pos2, new_amino_acid1, new_amino_acid2), 0)
        else:
            energy_new += J_dict.get((pos1, other_pos, new_amino_acid1, other_aa), 0)

    for other_pos in range(min_position, max_position+1):
        other_aa = seq[other_pos - min_position] 
        if other_pos == pos2:
            continue
        if other_pos == pos1:
            continue
        else:
            energy_new += J_dict.get((pos2, other_pos, new_amino_acid2, other_aa), 0)
    
    return energy_old - energy_new




## 3.3 flip on 1220

run de on all 1220 sequences with D148B, C140D (C140D, D148B), if the consensus vs one of 1220 sequences where the DE of each mutation changes its size compare to others  ( DE -DE sign change from consensus where the sign change means the sign of de D148B- deC140D sign change), record that sequence with its mutaitons and mutation len

make this a flip function, with mutation1 and 2 ans input and output the file in data/antag_out/{mut1}_{mut2}.tsv

In [125]:
import os

def flip(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with sign change
    sequences_with_sign_change = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)
    
    # Save the consensus energies to a TSV file######################################################
    consensus_output_dir = 'data/consensus_out'
    os.makedirs(consensus_output_dir, exist_ok=True)
    consensus_output_file = os.path.join(consensus_output_dir, f'{untranslated_mutation1}_{untranslated_mutation2}.tsv')

    consensus_data = {
        'Sequence': [consensus_sequence],  # Wrap the sequence in a list
        'Mutations': [[]],  # Wrap the empty list in another list
        'Mutation_count': [0],  # Wrap the integer in a list
        'dm1': [de_mutation1_consensus],
        'dm2': [de_mutation2_consensus],
        'dm12': [calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict)]
    }

    consensus_df = pd.DataFrame(consensus_data)
    consensus_df.to_csv(consensus_output_file, sep='\t', index=False)

    print(f"Consensus energies saved to {consensus_output_file}")
    ##################################################################################################

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']

        # Skip sequences without the specified mutations
        # if mutation1 not in mutations or mutation2 not in mutations:
        #     continue

        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for sign change
        if (de_mutation1 - de_mutation2) * (de_mutation1_consensus - de_mutation2_consensus) < 0 and dm12 is not None:
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2

            sequences_with_sign_change.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                # 'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
                # 'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
                'Dm1m2-max(dm1,dm2)': dm12 - max_dm1_dm2,
            })

    # Create a DataFrame to store the results
    sign_change_df = pd.DataFrame(sequences_with_sign_change)

    # Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
    if sign_change_df.empty:
        print(f"No sequences found for mutation pair: {mutation1}, {mutation2}")
    else:
        sign_change_df = sign_change_df.sort_values(by='Dm1m2-max(dm1,dm2)', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/flip_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{untranslated_mutation1}_{untranslated_mutation2}.tsv')

    # Save the results to a TSV file
    sign_change_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")

## 3.4 compensate on 1220

In [126]:

def compensate(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with compensation
    sequences_with_compensation = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']
        

        # Skip sequences without the specified mutations
        # if mutation1 not in mutations or mutation2 not in mutations:
        #     continue

        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for compensation
        if dm12 is not None and (dm12 > de_mutation1 or dm12 > de_mutation2) and not (dm12 > de_mutation1 and dm12 > de_mutation2):
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
            sequences_with_compensation.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
            })

    # Create a DataFrame to store the results
    compensation_df = pd.DataFrame(sequences_with_compensation)
    if compensation_df.empty:
        print(f"No sequences found for mutation pair: {mutation1}, {mutation2}")
    else:
        compensation_df = compensation_df.sort_values(by='Dm1m2-min(dm1,dm2)', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/compensate_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{untranslated_mutation1}_{untranslated_mutation2}.tsv')

    # Save the results to a TSV file
    compensation_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")


## 3.4.2 rescue

In [127]:

def rescue(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with compensation
    sequences_with_compensation = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']
        

        # Skip sequences without the specified mutations
        # if mutation1 not in mutations or mutation2 not in mutations:
        #     continue

        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for compensation
        if dm12 is not None and dm12 > de_mutation1 and dm12 > de_mutation2:
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
            sequences_with_compensation.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                'Dm1m2-max(dm1,dm2)': dm12 - max_dm1_dm2
            })

    # Create a DataFrame to store the results
    rescue_df = pd.DataFrame(sequences_with_compensation)
    if rescue_df.empty:
        print(f"No sequences found for mutation pair: {mutation1}, {mutation2}")
    else:
        rescue_df = rescue_df.sort_values(by='Dm1m2-max(dm1,dm2)', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/rescue_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{untranslated_mutation1}_{untranslated_mutation2}.tsv')

    # Save the results to a TSV file
    rescue_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")

## 3.5 antagonistic interactions

In [128]:
import os

def antagonistic(mutation1, mutation2 , untranslated_mutation1, untranslated_mutation2):
    # Extract position and amino acid information from the mutations
    pos1, old_aa1, new_aa1 = int(mutation1[1:-1]), mutation1[0], mutation1[-1]
    pos2, old_aa2, new_aa2 = int(mutation2[1:-1]), mutation2[0], mutation2[-1]

    # List to store sequences with antagonistic interactions
    sequences_with_antagonistic = []

    # Calculate delta E for the consensus sequence
    de_mutation1_consensus = calculate_delta_e(pos1, old_aa1, new_aa1, consensus_sequence, J_dict)
    de_mutation2_consensus = calculate_delta_e(pos2, old_aa2, new_aa2, consensus_sequence, J_dict)

    print("-" * 50)
    print(f'Consensus delta E for {mutation1}: {de_mutation1_consensus}, {mutation2}: {de_mutation2_consensus}')
    print('Consensus dm12:', calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, consensus_sequence, J_dict))
    print("-" * 50)

    # Iterate through all sequences
    for index, row in sequence_df.iterrows():
        sequence = row['Sequence']
        mutations = row['Mutations']
        mutation_count = row['Mutations_count']
        
        # Skip sequences without the specified mutations
        # if mutation1 not in mutations or mutation2 not in mutations:
        #     continue
        
        # Calculate delta E for the specified mutations
        de_mutation1 = calculate_delta_e(pos1, old_aa1, new_aa1, sequence, J_dict)
        de_mutation2 = calculate_delta_e(pos2, old_aa2, new_aa2, sequence, J_dict)
        dm12 = calculate_dm12(pos1, old_aa1, new_aa1, pos2, old_aa2, new_aa2, sequence, J_dict)

        # Check for antagonistic interaction
        if dm12 is not None and dm12 < de_mutation1 and dm12 < de_mutation2:
            min_dm1_dm2 = min(de_mutation1, de_mutation2)
            max_dm1_dm2 = max(de_mutation1, de_mutation2)
            max_divide_min_dm1_dm2 = max_dm1_dm2 / min_dm1_dm2
            
            sequences_with_antagonistic.append({
                'Sequence': sequence,
                'Mutations': mutations,
                'Mutation_count': mutation_count,
                'dm1': de_mutation1,
                'dm2': de_mutation2,
                'dm12': dm12,
                # 'Dm1m2-max/min(dm1,dm2)': dm12 - max_divide_min_dm1_dm2,
                # 'Dm1m2-min(dm1,dm2)': dm12 - min_dm1_dm2
                'Dm1m2-min(dm1,dm2)': dm12- min_dm1_dm2
            })

    # Create a DataFrame to store the results
    antagonistic_df = pd.DataFrame(sequences_with_antagonistic)

    # Sort the DataFrame by 'Dm1m2-min(dm1,dm2)' in descending order
    if antagonistic_df.empty:
        print(f"No sequences found for mutation pair: {mutation1}, {mutation2}")
    else:
        antagonistic_df = antagonistic_df.sort_values(by='Dm1m2-min(dm1,dm2)', ascending=False)

    # Define the output directory and file path
    output_dir = 'data/antag_out'
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'{untranslated_mutation1}_{untranslated_mutation2}.tsv')
    # If the mutation pair is G140S and Q148H, print dm12 values directly in output
    # if untranslated_mutation1 == 'G140S' and untranslated_mutation2 == 'Q148H':
    #     print(f"dm12 values for {untranslated_mutation1} and {untranslated_mutation2}:")
    #     for dm12_value in antagonistic_df['dm12'].tolist():
    #         print(dm12_value)
    # else:
    #     pass
    # Save the results to a TSV file
    antagonistic_df.to_csv(output_file, sep='\t', index=False)

    print(f"Results saved to {output_file}")


In [129]:
# # Define the mutations
# mutation1 = 'G140S'
# mutation2 = 'Q148H'
# Load the translated_top_syn_list.tsv file
translated_top_syn_list = pd.read_csv('data/translated_top_syn_list.tsv', sep='\t', comment = '#')

# Iterate through each row in the dataframe
for index, row in translated_top_syn_list.iterrows():
    # Extract mutation details
    mutation1 = f"{row['wildtype_aa1']}{row['pos1']}{row['mutate_aa1']}"
    mutation2 = f"{row['wildtype_aa2']}{row['pos2']}{row['mutate_aa2']}"
    untranslated_mutation1 = f"{row['untranslated_wildtype_aa1']}{row['pos1']}{row['untranslated_mutate_aa1']}"
    untranslated_mutation2 = f"{row['untranslated_wildtype_aa2']}{row['pos2']}{row['untranslated_mutate_aa2']}"
   

    # Print the mutations being processed
    print(f"Processing mutations: {mutation1}, {mutation2}")

    # Test the flip function
    print("Testing flip function:")
    flip(mutation1, mutation2,untranslated_mutation1, untranslated_mutation2)

    # Test the compensate function
    print("\nTesting compensate function:")
    compensate(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2)

    # Test the rescue function
    print("\nTesting rescue function:")
    rescue(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2)

    # Test the antagonistic function
    print("\nTesting antagonistic function:")
    antagonistic(mutation1, mutation2, untranslated_mutation1, untranslated_mutation2)

Processing mutations: C101A, D190C
Testing flip function:
--------------------------------------------------
Consensus delta E for C101A: -3.8616750240325928, D190C: -4.183330535888672
Consensus dm12: -4.7413273
--------------------------------------------------
Consensus energies saved to data/consensus_out/K101E_G190S.tsv
Results saved to data/flip_out/K101E_G190S.tsv

Testing compensate function:
--------------------------------------------------
Consensus delta E for C101A: -3.8616750240325928, D190C: -4.183330535888672
Consensus dm12: -4.7413273
--------------------------------------------------
Results saved to data/compensate_out/K101E_G190S.tsv

Testing rescue function:
--------------------------------------------------
Consensus delta E for C101A: -3.8616750240325928, D190C: -4.183330535888672
Consensus dm12: -4.7413273
--------------------------------------------------
Results saved to data/rescue_out/K101E_G190S.tsv

Testing antagonistic function:
---------------------------